In [ ]:
PRODUZIONE sub.set dati da ultimo aggiornamento microdati disponibile 2013-2025
ambito: ricerca su 'report II SEM 2025 -LAVORO IN SOMMINISTRAZIONE'
operazioni di ricodifica per ottenere un file df di lavoro 

#recode base della tipologia contrattuale
#recode qualifiche con qualifiche 5 digit cp2021 (nel 2025)
#recode ETA_LAV con nuovo campo ETA_LAV_ISTAT

In [6]:
#-----------------librerie
import pandas as pd
import numpy as np
import datetime
import chart_studio.plotly as py
import seaborn as sns
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"
# import matplotlib.pyplot as plt  #per export dei seaborn plot in image file
%run ../Libreria_JUPITER.ipynb
%run ../MICRODATI_funzioni.ipynb
from unicodedata import normalize
from datetime import date
import html5lib
import os
import re
from datetime import date
import xlsxwriter

In [2]:
#--------------------- SET
# consento un output multiplo nella stgessa cella 
%config InteractiveShell.ast_node_interactivity = 'all'
# definizione di un limite di viusalizzazione delle righe in output
pd.set_option('display.max_rows', 400) #None
# definizione di un limite di viusalizzazione delle righe in output
pd.set_option('display.max_columns', 80) #None

#definizione di un limite alla larghezza di una colonna (ritorno a capo)
pd.set_option('display.max_colwidth', None)#None

In [2]:
fold_source = 'C:\\Users\\lorid\\OneDrive\\Documenti\\Lavori_vari\\sistal_2\\eventi_SO'
fileTarget=fold_source+'\\Microdati2025_delta_11_03_2026__2013-2025_RICODIFICATI.csv'
df=pd.read_csv(fileTarget)

C:\Users\lorid\AppData\Local\Temp\ipykernel_18572\3987013027.py:3: DtypeWarning:

Columns (36,37,38) have mixed types. Specify dtype option on import or set low_memory=False.



In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1361191 entries, 0 to 1361190
Data columns (total 61 columns):
 #   Column                         Non-Null Count    Dtype  
---  ------                         --------------    -----  
 0   Unnamed: 0                     1361191 non-null  int64  
 1   ID_RAPPORTO                    1361191 non-null  object 
 2   DATA_EVENTO                    1361191 non-null  object 
 3   ANNO_EVENTO                    1361191 non-null  int64  
 4   MESE_EVENTO                    1361191 non-null  int64  
 5   TIPO_EVENTO                    1361191 non-null  object 
 6   CODICE_TRASFORMAZIONE          76371 non-null    object 
 7   CODICE_CAUSA_CESSAZIONE        517655 non-null   object 
 8   CODICE_FISCALE_DATORE          1361191 non-null  object 
 9   RAGSOCDATLAV                   1361191 non-null  object 
 10  CODICE_FISCALE_LAVORATORE      1361191 non-null  object 
 11  COD_TIPOLOGIA_CONTRATTUALE     1361191 non-null  object 
 12  COD_TIPO_ORARI

In [7]:
pd.crosstab?


Signature:
pd.crosstab(
    index,
    columns,
    values=None,
    rownames=None,
    colnames=None,
    aggfunc=None,
    margins: 'bool' = False,
    margins_name: 'Hashable' = 'All',
    dropna: 'bool' = True,
    normalize: 'bool' = False,
) -> 'DataFrame'
Docstring:
Compute a simple cross tabulation of two (or more) factors.

By default, computes a frequency table of the factors unless an
array of values and an aggregation function are passed.

Parameters
----------
index : array-like, Series, or list of arrays/Series
    Values to group by in the rows.
columns : array-like, Series, or list of arrays/Series
    Values to group by in the columns.
values : array-like, optional
    Array of values to aggregate according to the factors.
    Requires `aggfunc` be specified.
rownames : sequence, default None
    If passed, must match number of row arrays passed.
colnames : sequence, default None
    If passed, must match number of column arrays passed.
aggfunc : function, optional
   

In [5]:
df.shape?


Type:        property
String form: <property object at 0x000001DA4A313E00>
Docstring:  
Return a tuple representing the dimensionality of the DataFrame.

See Also
--------
ndarray.shape : Tuple of array dimensions.

Examples
--------
>>> df = pd.DataFrame({'col1': [1, 2], 'col2': [3, 4]})
>>> df.shape
(2, 2)

>>> df = pd.DataFrame({'col1': [1, 2], 'col2': [3, 4],
...                    'col3': [5, 6]})
>>> df.shape
(2, 3)


In [6]:
#set dati di base negli anni 2015-2025 con sede di lavoro SOndrio
df1525=df[df["ANNO_EVENTO"]>2014]
df1525SO=df1525[df1525["nome_provincia_sede_lavoro"]=="SONDRIO"]
df1525SO.shape

(1041726, 61)

In [ ]:
########################################
        CONTROLLI SOLO CONTROLLI
########################################

In [11]:
pd.crosstab(
    df1525SO.ANNO_EVENTO,
    df1525SO.TIPO_EVENTO,
    margins =True
) 

TIPO_EVENTO,A,C,P,T,All
ANNO_EVENTO,,,,,
2015,31033,30477,14906,4116,80532
2016,29509,28034,14524,4078,76145
2017,35934,33731,18082,4469,92216
2018,38520,36658,19694,5512,100384
2019,37731,37123,16612,6179,97645
2020,28604,32018,16202,5021,81845
2021,37642,32564,18171,4638,93015
2022,39887,39246,20124,6027,105284
2023,40346,38551,18132,6142,103171


In [25]:
a=df1525SO[df1525SO.TIPO_EVENTO=='A']
frequenze = a['COD_TIPOLOGIA_CONTRATTUALE'].value_counts()
frequenze

COD_TIPOLOGIA_CONTRATTUALE
A.02.00    230011
A.01.00     43176
A.05.01     35187
A.06.01     30835
A.02.01     16316
A.03.09     15576
A.04.00      7149
C.01.00      6946
G.03.00      4772
B.03.00      3274
B.04.00      2406
A.05.00      2141
A.04.01      1446
A.06.00      1217
C.03.00      1001
B.01.00       213
A.03.08       168
A.08.01       115
AP-USOM       106
I.01.00        50
B.02.00        42
AP-ULAV        39
H.02.00        27
A.08.00        24
C.04.00        22
N.02.00         8
L.01.01         6
M.01.00         5
A.03.07         5
A.03.10         5
N.01.00         4
N.03.00         4
A.03.02         2
H.03.00         2
L.01.00         1
A.07.00         1
Name: count, dtype: int64

In [24]:
#COD_TIPOLOGIA_CONTRATTUALE solo per avviamenti
a=df1525SO[df1525SO.TIPO_EVENTO=='A']

pd.crosstab(
    a.ANNO_EVENTO,
    a.COD_TIPOLOGIA_CONTRATTUALE,
    margins =True
) 


COD_TIPOLOGIA_CONTRATTUALE,A.01.00,A.02.00,A.02.01,A.03.02,A.03.07,A.03.08,A.03.09,A.03.10,A.04.00,A.04.01,...,H.02.00,H.03.00,I.01.00,L.01.00,L.01.01,M.01.00,N.01.00,N.02.00,N.03.00,All
ANNO_EVENTO,,,,,,,,,,,,,,,,,,,,,
2015,5290,15641,2497,1,0,6,930,1,650,123,...,0,0,6,1,6,3,0,0,0,31033
2016,3595,16457,1515,0,0,11,1328,0,625,135,...,0,0,2,0,0,0,0,0,0,29509
2017,3190,19868,1442,0,1,10,1511,0,639,165,...,0,0,3,0,0,1,0,0,0,35934
2018,3941,21354,1342,0,1,20,1765,0,660,177,...,0,0,3,0,0,0,0,0,0,38520
2019,4182,21540,1488,0,0,16,1779,0,661,127,...,0,0,4,0,0,0,0,0,0,37731
2020,3313,15358,1055,0,2,10,1254,0,978,159,...,3,0,5,0,0,0,0,1,0,28604
2021,3237,21561,1368,0,1,15,1595,0,652,132,...,5,0,9,0,0,0,0,1,1,37642
2022,4146,23849,1539,1,0,16,1579,2,618,102,...,11,0,3,0,0,0,0,1,1,39887
2023,4223,24153,1255,0,0,29,1428,1,563,99,...,2,0,4,0,0,1,2,1,0,40346


In [27]:



a=df1525SO[df1525SO.TIPO_EVENTO=='A']

dir_lavoro = 'C:\\Users\\lorid\\OneDrive\\Documenti\\Lavori_vari\\sistal_2\\Domini'
target_fileTOT = fr"{dir_lavoro}\\SL2_DOM_TIPO_CONTRATTO.xlsx" 

#lettura dominio
df_lookup=pd.read_excel(target_fileTOT) #

#campi COD_TIPOLOGIA_CONTRATTUALE,DES_TIPOLOGIA_CONTRATTUALE

dizionario_mappa = dict(zip(df_lookup['COD_TIPOLOGIA_CONTRATTUALE'], df_lookup['DES_TIPOLOGIA_CONTRATTUALE']))
a['TIPOLOGIA_CONTRATTUALE'] = a['COD_TIPOLOGIA_CONTRATTUALE'].map(dizionario_mappa)

#----------------------------------------------

# Creazione di una tabella riassuntiva
tabella_freq = a['TIPOLOGIA_CONTRATTUALE'].value_counts().reset_index()
tabella_freq.columns = ['Valore', 'Conteggio']

# Aggiunta della percentuale
tabella_freq['Percentuale'] = (tabella_freq['Conteggio'] / tabella_freq['Conteggio'].sum()) * 100

tabella_freq


C:\Users\lorid\AppData\Local\Temp\ipykernel_18572\2876264858.py:12: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,Valore,Conteggio,Percentuale
0,Lavoro a tempo determinato,230011,57.173715
1,Lavoro a tempo indeterminato,43176,10.732236
2,Lavoro intermittente a tempo determinato,35187,8.746414
3,Lavoro interinale (o a scopo di somministrazio...,30835,7.664640
4,Lavoro a tempo determinato per sostituzione,16316,4.055660
5,Apprendistato professionalizzante o contratto ...,15576,3.871718
6,Lavoro domestico a tempo indeterminato,7149,1.777023
7,Tirocinio,6946,1.726564
8,Lavoro autonomo nello spettacolo,4772,1.186174
9,Collaborazione coordinata e continuativa,3274,0.813816


In [ ]:
#------------------------------FINE CONTROLLI

In [ ]:
########################################
        RICODIFICHE
########################################

In [ ]:
#############################################
        PRODUZIONE df di lavoro
#############################################

In [ ]:
#recode ETA_LAV con nuovo campo ETA_LAV_ISTAT

In [29]:
#recode base della tipologia contrattuale
#save 2015-2025
#nome:MICRO_SO_1525.csv


a=df1525SO.copy()
dir_save_dati= "C:\\Users\\lorid\\OneDrive\\Documenti\\Lavori_vari\\sistal_2\\eventi_SO\\eventi_SO_somministrazione_2025"
target_file_save=fr"{dir_save_dati}\\MICRO_SO_1525.csv"

##################  recode con dominio aggiornato
dir_lavoro = 'C:\\Users\\lorid\\OneDrive\\Documenti\\Lavori_vari\\sistal_2\\Domini'
target_fileTOT = fr"{dir_lavoro}\\SL2_DOM_TIPO_CONTRATTO.xlsx" 

#lettura dominio
df_lookup=pd.read_excel(target_fileTOT) #

#campi COD_TIPOLOGIA_CONTRATTUALE,DES_TIPOLOGIA_CONTRATTUALE

dizionario_mappa = dict(zip(df_lookup['COD_TIPOLOGIA_CONTRATTUALE'], df_lookup['DES_TIPOLOGIA_CONTRATTUALE']))
a['TIPOLOGIA_CONTRATTUALE'] = a['COD_TIPOLOGIA_CONTRATTUALE'].map(dizionario_mappa)

#----------------------------------------------

###############  save

a.to_csv(target_file_save)





In [40]:

#save solo 2025
#nome:MICRO_SO_1525_5digit.csv

dir_save_dati= "C:\\Users\\lorid\\OneDrive\\Documenti\\Lavori_vari\\sistal_2\\eventi_SO\\eventi_SO_somministrazione_2025"
target_file_read=fr"{dir_save_dati}\\MICRO_SO_1525.csv"
df=pd.read_csv(target_file_read)

dir_save_dati= "C:\\Users\\lorid\\OneDrive\\Documenti\\Lavori_vari\\sistal_2\\eventi_SO\\eventi_SO_somministrazione_2025"
target_file_save=fr"{dir_save_dati}\\MICRO_SO_25.csv"

b=df[df.ANNO_EVENTO==2025]

###############  save

b.to_csv(target_file_save)



C:\Users\lorid\AppData\Local\Temp\ipykernel_18572\4161714828.py:7: DtypeWarning:

Columns (37,38,39) have mixed types. Specify dtype option on import or set low_memory=False.



In [6]:
dir_save_dati= "C:\\Users\\lorid\\OneDrive\\Documenti\\Lavori_vari\\sistal_2\\eventi_SO\\eventi_SO_somministrazione_2025"
target_file_save=fr"{dir_save_dati}\\MICRO_SO_1525.csv"
vedi=pd.read_csv(target_file_save)
vedi.shape

C:\Users\lorid\AppData\Local\Temp\ipykernel_7020\3262840706.py:3: DtypeWarning:

Columns (37,38,39) have mixed types. Specify dtype option on import or set low_memory=False.



(1041726, 63)

In [ ]:
#recode qualifiche con qualifiche 5 digit cp2021 (nel 2025)

In [12]:
############### corretta

#recode qualifiche con qualifiche 5 digit cp2021 (nel 2025)
#nome:MICRO_SO_1525.csv


# ricodifica delle qualifiche a 5digit
dir_save_dati= "C:\\Users\\lorid\\OneDrive\\Documenti\\Lavori_vari\\sistal_2\\eventi_SO\\eventi_SO_somministrazione_2025"
target_file_save=fr"{dir_save_dati}\\MICRO_SO_1525.csv"
df=pd.read_csv(target_file_save)

#--- sistemo il passaggio a 5 digit per verifica di nomi doppi riscontrati
# produco una ricodifica 

target_file=fr"{dir_save_dati}\\Tabella di raccordo Istat CP2011-CP2021 6 digit.csv"

df_5digit = pd.read_csv(target_file)
#df_5digit

#


C:\Users\lorid\AppData\Local\Temp\ipykernel_7020\3344590228.py:10: DtypeWarning:

Columns (37,38,39) have mixed types. Specify dtype option on import or set low_memory=False.



In [13]:
import pandas as pd

def tronca_al_quinto_punto(codice):
    if pd.isna(codice):
        return None
    
    # Dividiamo la stringa in base a TUTTI i punti presenti
    parti = str(codice).split('.')
    
    # Se ci sono almeno 5 parti, le prendiamo e le ricongiungiamo con il punto
    if len(parti) >= 5:
        return ".".join(parti[:5])
    
    # Se il codice è già più corto (meno di 5 sezioni), lo restituiamo così com'è
    return ".".join(parti)

# Applichiamo la funzione per creare il nuovo campo
df['COD_QUALIFICA_PROF_ISTAT_5DGT'] = df['COD_QUALIFICA_PROF_ISTAT_COB'].apply(tronca_al_quinto_punto)

# --- CONTROLLO DI QUALITÀ ---
# Verifichiamo se ci sono codici che non rispettano lo standard (es. troppo corti)
anomalie = df[df['COD_QUALIFICA_PROF_ISTAT_5DGT'].str.count('\.') < 4]

if not anomalie.empty:
    print(f"Attenzione: trovati {len(anomalie)} codici con meno di 5 digit.")
    print(anomalie[['COD_QUALIFICA_PROF_ISTAT_COB', 'COD_QUALIFICA_PROF_ISTAT_5DGT']].head())
else:
    print("Tutti i codici sono stati troncati correttamente al 5° livello (4 punti separatori).")

# Visualizzazione controllo
print(df[['COD_QUALIFICA_PROF_ISTAT_COB', 'COD_QUALIFICA_PROF_ISTAT_5DGT']].head())

Tutti i codici sono stati troncati correttamente al 5° livello (4 punti separatori).
  COD_QUALIFICA_PROF_ISTAT_COB COD_QUALIFICA_PROF_ISTAT_5DGT
0                  6.1.4.1.1.3                     6.1.4.1.1
1                  2.5.5.1.4.9                     2.5.5.1.4
2                  8.1.4.2.0.4                     8.1.4.2.0
3                  3.4.2.5.1.0                     3.4.2.5.1
4                  5.4.3.1.0.0                     5.4.3.1.0


In [14]:


# 1. Pulizia della tabella di raccordo: teniamo solo una riga per ogni codice
# Rimuoviamo i duplicati basandoci sulla colonna 'Codice CP2021'
df_5digit_unique = df_5digit.drop_duplicates(subset=['Codice CP2021'])

# 2. Eseguiamo il merge usando la tabella pulita
df_finale = df.merge(
    df_5digit_unique[['Codice CP2021', 'Descrizione CP2021']], 
    left_on='COD_QUALIFICA_PROF_ISTAT_5DGT', 
    right_on='Codice CP2021', 
    how='left'
)


# Rinominiamo la colonna appena acquisita nel nome desiderato
df_finale = df_finale.rename(columns={'Descrizione CP2021': 'qualifiche_cp2021_5digit'})

# Rimuoviamo la colonna 'Codice CP2021' che è diventata ridondante dopo il merge
df_finale = df_finale.drop(columns=['Codice CP2021'])

# 3. Verifica immediata dei record
print(f"Record originali: {len(df)}")
print(f"Record dopo merge pulito: {len(df_finale)}")

Record originali: 1041726
Record dopo merge pulito: 1041726


In [15]:
df_finale[['qualifiche_cp2021','qualifiche_cp2021_5digit']].head(10)

,qualifiche_cp2021,qualifiche_cp2021_5digit
0,imbianchino,Pittori edili
1,grafico creativo,Creatori artistici a fini commerciali (esclusa...
2,garzone di cucina,Personale non qualificato nei servizi di risto...
3,Organizzatori di eventi e di strutture sportive,Organizzatori di eventi e di strutture sportive
4,Acconciatori,NaN
5,cassiere di pubblico esercizio,Cassieri di esercizi commerciali
6,addetto all'ufficio prenotazioni in agenzia di...,Addetti agli sportelli delle agenzie di viaggio
7,confezionatore caseario artigianale,Artigiani ed operai specializzati delle lavora...
8,Gelatai,Gelatai
9,addetto alla pulizia delle camere,Personale non qualificato addetto alla pulizia...


In [19]:
# Sistemo i NaN nelle qualifiche_cp2021_5digit 
# ora sistemiamo tutti i cp2011 che hanno cambiato codice nel cp2021 e
# che appaiono ancora nella vecchia formulazione 

# 1. Preparazione tabella di recupero (CP2011 -> Descrizione)
# Prendiamo solo le colonne necessarie e rinominiamo subito la descrizione per il recupero
df_raccordo_2011 = df_5digit[['Codice CP2011', 'Descrizione CP2021']].drop_duplicates(subset=['Codice CP2011'])
df_raccordo_2011 = df_raccordo_2011.rename(columns={'Descrizione CP2021': 'descrizione_recovery'})

# Pulizia stringhe per la chiave di raccordo
df_raccordo_2011['Codice CP2011'] = df_raccordo_2011['Codice CP2011'].astype(str).str.strip()

# 2. Esecuzione del merge di recupero
# Usiamo il codice COB originale del tuo DF principale come chiave di confronto
df_finale = df_finale.merge(
    df_raccordo_2011, 
    left_on='COD_QUALIFICA_PROF_ISTAT_COB', 
    right_on='Codice CP2011', 
    how='left'
)

# 3. Riempimento dei NaN
# Se la colonna principale è NaN, prendiamo il valore dalla colonna di recupero
df_finale['qualifiche_cp2021_5digit'] = df_finale['qualifiche_cp2021_5digit'].fillna(df_finale['descrizione_recovery'])

# 4. Pulizia finale delle colonne tecniche
colonne_da_rimuovere = [c for c in ['Codice CP2011', 'descrizione_recovery'] if c in df_finale.columns]
df_finale = df_finale.drop(columns=colonne_da_rimuovere)

# --- VERIFICA DEI RISULTATI ---
rimasti_nan = df_finale['qualifiche_cp2021_5digit'].isna().sum()
print(f"Record totali: {len(df_finale)}")
print(f"NaN residui dopo il recupero tramite CP2011: {rimasti_nan}")

# Controllo specifico per gli acconciatori
if 'qualifiche_cp2021' in df_finale.columns:
    mask = df_finale['qualifiche_cp2021'].str.contains('Acconciatore', na=False, case=False)
    print("\n--- Controllo Qualifiche Recuperate (Acconciatori) ---")
    print(df_finale[mask][['COD_QUALIFICA_PROF_ISTAT_COB', 'qualifiche_cp2021_5digit']].drop_duplicates())

Record totali: 1041726
NaN residui dopo il recupero tramite CP2011: 28

--- Controllo Qualifiche Recuperate (Acconciatori) ---
      COD_QUALIFICA_PROF_ISTAT_COB qualifiche_cp2021_5digit
17626                  5.4.3.1.0.1             Acconciatori


In [21]:
df_finale[['qualifiche_cp2021','qualifiche_cp2021_5digit']].head(50)

,qualifiche_cp2021,qualifiche_cp2021_5digit
0,imbianchino,Pittori edili
1,grafico creativo,Creatori artistici a fini commerciali (esclusa...
2,garzone di cucina,Personale non qualificato nei servizi di risto...
3,Organizzatori di eventi e di strutture sportive,Organizzatori di eventi e di strutture sportive
4,Acconciatori,Acconciatori
5,cassiere di pubblico esercizio,Cassieri di esercizi commerciali
6,addetto all'ufficio prenotazioni in agenzia di...,Addetti agli sportelli delle agenzie di viaggio
7,confezionatore caseario artigianale,Artigiani ed operai specializzati delle lavora...
8,Gelatai,Gelatai
9,addetto alla pulizia delle camere,Personale non qualificato addetto alla pulizia...


In [22]:
# salvataggio file generale con qualifiche 5 digit 
# in teoria da usare in maniera privilegiata

target_file_save=fr"{dir_save_dati}\\MICRO_SO_1525_5digit.csv"


###############  save

df_finale.to_csv(target_file_save)

In [ ]:
#recode ETA_LAV con nuovo campo ETA_LAV_ISTAT

In [17]:
#recode ETA_LAV con nuovo campo ETA_LAV_ISTAT
#nome:MICRO_SO_1525.csv


# ricodifica delle qualifiche a 5digit
dir_save_dati= "C:\\Users\\lorid\\OneDrive\\Documenti\\Lavori_vari\\sistal_2\\eventi_SO\\eventi_SO_somministrazione_2025"
target_file_save=fr"{dir_save_dati}\\MICRO_SO_1525_5digit.csv"
df=pd.read_csv(target_file_save)

C:\Users\lorid\AppData\Local\Temp\ipykernel_7204\2309910035.py:8: DtypeWarning:

Columns (39,40,41) have mixed types. Specify dtype option on import or set low_memory=False.



In [18]:
# 1. Definiamo i punti di taglio (i limiti delle fasce)
# Usiamo 0 come limite minimo di sicurezza e 100 come massimo.
limiti_eta = [0, 24, 34, 44, 54, 64, 82,150]

# 2. Definiamo le etichette esatte per ogni "cesto"
etichette_istat = ['15-24 anni', '25-34 anni', '35-44 anni', '45-54 anni', '55-64 anni', '65-82 anni','82+ anni']

# 3. Creiamo la nuova colonna applicando pd.cut
df['ETA_LAV_ISTAT'] = pd.cut(
    df['ETA_LAV'], 
    bins=limiti_eta, 
    labels=etichette_istat, 
    right=True # Significa che il limite superiore è incluso (es. fino a 24 compreso)
)

In [19]:
freq(df,'ETA_LAV_ISTAT','ETA_LAV_ISTAT')

ETA_LAV_ISTAT,freq,%
25-34 anni,275008,26.4
15-24 anni,228327,21.9
35-44 anni,214379,20.6
45-54 anni,196089,18.8
55-64 anni,111465,10.7
65-82 anni,16413,1.6
82+ anni,43,0.0
Total,1041724,100.0


In [20]:
# 1. Creiamo la crosstab incrociando l'età puntuale (righe) con le fasce (colonne)
verifica_eta = pd.crosstab(df['ETA_LAV'], df['ETA_LAV_ISTAT'])

# 2. Stampiamo un segmento strategico per vedere il "salto" di fascia
# Guardiamo le età dal 22 al 27 per assicurarci che il confine del 24 funzioni
print(verifica_eta.loc[22:27])

ETA_LAV_ISTAT  15-24 anni  25-34 anni  35-44 anni  45-54 anni  55-64 anni  \
ETA_LAV                                                                     
22                  34032           0           0           0           0   
23                  33988           0           0           0           0   
24                  33714           0           0           0           0   
25                      0       33115           0           0           0   
26                      0       32028           0           0           0   
27                      0       30448           0           0           0   

ETA_LAV_ISTAT  65-82 anni  82+ anni  
ETA_LAV                              
22                      0         0  
23                      0         0  
24                      0         0  
25                      0         0  
26                      0         0  
27                      0         0  


In [21]:
df.to_csv(target_file_save)

In [ ]:
#----------------fine recode eta

In [ ]:
controlli vari

In [17]:
# Estrazione dei record non associati
df_nan = df_finale[df_finale['qualifiche_cp2021_5digit'].isna()]

print(f"Record totali: {len(df_finale)}")
print(f"Record con NaN in nuova qualifica: {len(df_nan)}")

if not df_nan.empty:
    print("\n--- ANALISI CODICI MANCANTI (Primi 20 casi) ---")
    # Raggruppiamo per vedere quali codici e quali vecchie descrizioni generano NaN
    report_nan = df_nan.groupby(['COD_QUALIFICA_PROF_ISTAT_5DGT', 'qualifiche_cp2021']).size().reset_index(name='Conteggio')
    print(report_nan.sort_values(by='Conteggio', ascending=False).head(20))

    # Focus specifico su Acconciatore
    print("\n--- FOCUS: CASI 'ACCONCIATORE' CON NaN ---")
    acconciatori = df_nan[df_nan['qualifiche_cp2021'].str.contains('Acconciatore', na=False, case=False)]
    print(acconciatori[['COD_QUALIFICA_PROF_ISTAT_COB', 'COD_QUALIFICA_PROF_ISTAT_5DGT', 'qualifiche_cp2021']].drop_duplicates())

Record totali: 1041726
Record con NaN in nuova qualifica: 39656

--- ANALISI CODICI MANCANTI (Primi 20 casi) ---
    COD_QUALIFICA_PROF_ISTAT_5DGT  \
140                     5.4.4.3.0   
142                     5.4.4.3.0   
89                      5.4.2.1.4   
108                     5.4.3.2.0   
103                     5.4.3.1.0   
132                     5.4.4.3.0   
124                     5.4.4.2.0   
135                     5.4.4.3.0   
127                     5.4.4.2.0   
81                      3.4.2.2.0   
179                     5.4.8.7.0   
96                      5.4.3.1.0   
120                     5.4.3.3.0   
80                      3.4.2.2.0   
115                     5.4.3.3.0   
154                     5.4.7.2.0   
126                     5.4.4.2.0   
175                     5.4.8.6.0   
165                     5.4.8.2.0   
174                     5.4.8.6.0   

                                     qualifiche_cp2021  Conteggio  
140                                      

In [34]:

# per controlli


dir_save_dati= "C:\\Users\\lorid\\OneDrive\\Documenti\\Lavori_vari\\sistal_2\\eventi_SO\\eventi_SO_somministrazione_2025"
target_file_save=fr"{dir_save_dati}\\MICRO_SO_1525.csv"
df=pd.read_csv(target_file_save)




b=df[df.TIPO_EVENTO=='A']
# Creazione di una tabella riassuntiva
tabella_freq = b['TIPOLOGIA_CONTRATTUALE'].value_counts().reset_index()
tabella_freq.columns = ['Valore', 'Conteggio']

# Aggiunta della percentuale
tabella_freq['Percentuale'] = (tabella_freq['Conteggio'] / tabella_freq['Conteggio'].sum()) * 100

tabella_freq

C:\Users\lorid\AppData\Local\Temp\ipykernel_18572\3749434784.py:3: DtypeWarning:

Columns (37,38,39) have mixed types. Specify dtype option on import or set low_memory=False.



,Valore,Conteggio,Percentuale
0,Lavoro a tempo determinato,230011,57.173715
1,Lavoro a tempo indeterminato,43176,10.732236
2,Lavoro intermittente a tempo determinato,35187,8.746414
3,Lavoro interinale (o a scopo di somministrazio...,30835,7.664640
4,Lavoro a tempo determinato per sostituzione,16316,4.055660
5,Apprendistato professionalizzante o contratto ...,15576,3.871718
6,Lavoro domestico a tempo indeterminato,7149,1.777023
7,Tirocinio,6946,1.726564
8,Lavoro autonomo nello spettacolo,4772,1.186174
9,Collaborazione coordinata e continuativa,3274,0.813816
